In [11]:
# Task 1: Load dataset

import pandas as pd, numpy as np, glob, os
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

C = "../data/cleaned"
df = {}
for f in sorted(glob.glob(os.path.join(C, "*_clean.csv"))):
    name = os.path.basename(f).replace("_clean.csv", "")
    cols = pd.read_csv(f, nrows=0).columns
    date_cols = [c for c in cols if "date" in c.lower()]
    df[name] = pd.read_csv(f, parse_dates=date_cols)

print(f"Loaded {len(df)} cleaned tables")

Loaded 12 cleaned tables


In [12]:
#Task 2: Explore dataset

display(pd.DataFrame({
    "table": list(df),
    "rows": [d.shape[0] for d in df.values()],
    "columns": [d.shape[1] for d in df.values()],
}))

for name in ["sales_orders", "purchase_orders", "purchase_order_lines",
             "shipment_tracking", "stock_movements", "warehouse_inventory"]:
    d = df[name]
    print(f"\n===== {name} =====")
    display(d.head(3))
    print(d.dtypes.to_string())
    num = d.select_dtypes("number")
    if not num.empty:
        display(num.describe().round(2))

,table,rows,columns
0,carriers,4,2
1,fulfillment_status,4446,4
2,purchase_order_lines,2714,6
3,purchase_orders,1069,6
4,raw_materials,4,4
5,sales_orders,4359,6
6,shipment_tracking,4606,6
7,stock_movements,526,6
8,supplier_performance,4,5
9,suppliers,4,3



===== sales_orders =====


,SalesOrderID,OrderDate,ProductID,ProductName,Quantity,CustomerID
0,SO-9001,2025-11-29,P101,Magical Dark Chocolate,199.0,CUST-1304
1,SO-9002,2025-06-24,P102,Enchanted Milk Chocolate,120.0,CUST-1410
2,SO-9003,2026-01-27,P103,Golden Cocoa Truffles,154.0,CUST-1450


SalesOrderID            object
OrderDate       datetime64[ns]
ProductID               object
ProductName             object
Quantity               float64
CustomerID              object


,Quantity
count,4359.00
mean,104.95
std,54.86
min,10.00
25%,58.00
50%,105.00
75%,153.00
max,200.00



===== purchase_orders =====


... (13 outputs left out)


===== warehouse_inventory =====


,InventoryID,MaterialID,WarehouseID,QuantityOnHand,LastStocktakeDate
0,INV-1,RM01,WH-E,625,2026-11-05
1,INV-2,RM01,WH-N,2219,2026-10-21
2,INV-3,RM01,WH-A,4613,2026-11-04


InventoryID                  object
MaterialID                   object
WarehouseID                  object
QuantityOnHand                int64
LastStocktakeDate    datetime64[ns]


,QuantityOnHand
count,12.00
mean,4553.58
std,3265.99
min,625.00
25%,2141.25
50%,3174.50
75%,7423.25
max,9828.00


In [13]:
#Task 3 and 4: Verify missing values and duplicates

check = pd.DataFrame({
    "table": list(df),
    "rows": [len(d) for d in df.values()],
    "missing_cells": [int(d.isna().sum().sum()) for d in df.values()],
    "duplicate_rows": [int(d.duplicated().sum()) for d in df.values()],
})
display(check)

print("\nColumns that still have gaps (kept on purpose):")
for name, d in df.items():
    gaps = d.isna().sum()
    gaps = gaps[gaps > 0]
    if len(gaps):
        print(f"  {name}: {gaps.to_dict()}")

,table,rows,missing_cells,duplicate_rows
0,carriers,4,0,0
1,fulfillment_status,4446,181,0
2,purchase_order_lines,2714,113,6
3,purchase_orders,1069,89,0
4,raw_materials,4,0,0
5,sales_orders,4359,0,13
6,shipment_tracking,4606,176,0
7,stock_movements,526,24,0
8,supplier_performance,4,1,0
9,suppliers,4,0,0



Columns that still have gaps (kept on purpose):
  fulfillment_status: {'FulfillmentID': 181}
  purchase_order_lines: {'POLineID': 113}
  purchase_orders: {'ExpectedDeliveryDate': 42, 'ReportedTotalCost': 47}
  shipment_tracking: {'ShipDate': 176}
  stock_movements: {'MovementID': 24}
  supplier_performance: {'QualityScore': 1}


In [14]:
#Task 5: Analyze inventory levels

mat = df["raw_materials"]
wh = df["warehouses"]
pol = df["purchase_order_lines"]

inv = (df["warehouse_inventory"]
       .merge(mat[["MaterialID", "MaterialName", "UnitCost"]], on="MaterialID", how="left")
       .merge(wh[["WarehouseID", "WarehouseName"]], on="WarehouseID", how="left"))
inv["StockValue"] = (inv["QuantityOnHand"] * inv["UnitCost"]).round(2)

by_mat = inv.groupby("MaterialName").agg(
    units_on_hand=("QuantityOnHand", "sum"),
    stock_value=("StockValue", "sum"))
by_mat["share_of_units_%"] = (by_mat["units_on_hand"] / by_mat["units_on_hand"].sum() * 100).round(1)
by_mat["share_of_value_%"] = (by_mat["stock_value"] / by_mat["stock_value"].sum() * 100).round(1)
print("--- Stock by material ---")
display(by_mat.sort_values("stock_value", ascending=False))

print("--- Units on hand: material x warehouse ---")
display(inv.pivot_table(index="MaterialName", columns="WarehouseName",
                        values="QuantityOnHand", aggfunc="sum", margins=True))

print("--- 3 lowest stock positions ---")
display(inv.nsmallest(3, "QuantityOnHand")[
    ["MaterialName", "WarehouseName", "QuantityOnHand", "LastStocktakeDate"]])

print("--- Purchased per material (from PO lines) ---")
bought = (pol.groupby("MaterialID")
          .agg(units_purchased=("Quantity", "sum"),
               avg_unit_cost=("UnitCost", "mean"),
               spend=("LineTotal", "sum"))
          .round(2).reset_index()
          .merge(mat[["MaterialID", "MaterialName"]], on="MaterialID")
          .set_index("MaterialName").drop(columns="MaterialID"))
display(bought)

--- Stock by material ---


,units_on_hand,stock_value,share_of_units_%,share_of_value_%
MaterialName,,,,
Ghana Cocoa Beans,7457,152868.5,13.6,40.5
Whole Milk Powder,13538,118457.5,24.8,31.4
Organic Cane Sugar,19377,98822.7,35.5,26.2
Foil Wrappers,14271,7135.5,26.1,1.9


--- Units on hand: material x warehouse ---


WarehouseName,Asia-Pacific Hub,Euro Hub,North America Hub,All
MaterialName,,,,
Foil Wrappers,9828,1974,2469,14271
Ghana Cocoa Beans,4613,625,2219,7457
Organic Cane Sugar,7098,3880,8399,19377
Whole Milk Powder,1938,9403,2197,13538
All,23477,15882,15284,54643


--- 3 lowest stock positions ---


,MaterialName,WarehouseName,QuantityOnHand,LastStocktakeDate
0,Ghana Cocoa Beans,Euro Hub,625,2026-11-05
8,Whole Milk Powder,Asia-Pacific Hub,1938,2026-11-09
9,Foil Wrappers,Euro Hub,1974,2026-10-29


--- Purchased per material (from PO lines) ---


,units_purchased,avg_unit_cost,spend
MaterialName,,,
Ghana Cocoa Beans,821381.0,20.50,16838310.50
Organic Cane Sugar,924853.0,5.10,4716750.30
Whole Milk Powder,824461.0,8.75,7214033.75
Foil Wrappers,846046.0,0.50,423023.00


In [15]:
#Task 6: Analyze supplier performance

po = df["purchase_orders"].copy()
po["PlannedLeadDays"] = (po["ExpectedDeliveryDate"] - po["OrderDate"]).dt.days

po_lines = pol.merge(po[["PO_ID", "SupplierID"]].drop_duplicates("PO_ID"),
                     on="PO_ID", how="inner")
print(f"PO lines linked to a purchase order: {len(po_lines)} of {len(pol)}")

spend = po_lines.groupby("SupplierID").agg(
    po_lines=("Quantity", "size"),
    units_ordered=("Quantity", "sum"),
    spend=("LineTotal", "sum"))

orders = po.groupby("SupplierID").agg(
    purchase_orders=("PO_ID", "nunique"),
    avg_planned_lead_days=("PlannedLeadDays", "mean"))

perf = (df["suppliers"]
        .merge(df["supplier_performance"].drop(columns="SupplierName"), on="SupplierID", how="left")
        .merge(orders, on="SupplierID", how="left")
        .merge(spend, on="SupplierID", how="left"))
perf["spend_share_%"] = (perf["spend"] / perf["spend"].sum() * 100).round(1)
display(perf.sort_values("spend", ascending=False).round(2))

print("--- Purchase order status by supplier ---")
display(pd.crosstab(po["SupplierID"], po["Status"]))

PO lines linked to a purchase order: 2293 of 2714


,SupplierID,SupplierName,country,OnTimeDeliveryRate,QualityScore,AverageLeadTime_Days,purchase_orders,avg_planned_lead_days,po_lines,units_ordered,spend,spend_share_%
1,SUP03,Dairy Farms United,Germany,0.98,4.36,32,279,29.47,632,796484.0,6871076.10,28.0
0,SUP01,Cocoa World Inc.,Ghana,0.95,4.33,19,261,29.82,565,699953.0,6106407.00,24.9
2,SUP04,Packaging Pros,China,0.88,4.78,22,249,30.27,545,686105.0,5833926.55,23.7
3,SUP02,Global Sweeteners,Unknown,0.99,NaN,35,273,28.18,551,703023.0,5759337.30,23.4


--- Purchase order status by supplier ---


Status,Cancelled,Delivered,Ordered,Shipped,Unknown
SupplierID,,,,,
SUP01,71,48,66,68,10
SUP02,57,54,71,74,18
SUP03,76,58,72,61,13
SUP04,42,69,65,65,11


In [16]:
#Task 7: Analyze warehouse stock

mv = (df["stock_movements"]
      .merge(mat[["MaterialID", "MaterialName"]], on="MaterialID", how="left")
      .merge(wh[["WarehouseID", "WarehouseName"]], on="WarehouseID", how="left"))

wh_stock = inv.groupby("WarehouseName").agg(
    units_on_hand=("QuantityOnHand", "sum"),
    stock_value=("StockValue", "sum"))
wh_stock["share_of_stock_%"] = (wh_stock["units_on_hand"] / wh_stock["units_on_hand"].sum() * 100).round(1)

inbound = mv.groupby("WarehouseName").agg(
    inbound_movements=("MovementID", "size"),
    inbound_units=("Quantity", "sum"))

print("--- Stock on hand and inbound by warehouse ---")
display(wh_stock.join(inbound))

print("--- Inbound units by month and warehouse (last 12 months) ---")
mv["Month"] = mv["Date"].dt.to_period("M")
display(mv.pivot_table(index="Month", columns="WarehouseName",
                       values="Quantity", aggfunc="sum").tail(12))

--- Stock on hand and inbound by warehouse ---


,units_on_hand,stock_value,share_of_stock_%,inbound_movements,inbound_units
WarehouseName,,,,,
Asia-Pacific Hub,23477,152637.80,43.0,182,230090.0
Euro Hub,15882,115863.75,29.1,190,240485.0
North America Hub,15284,108782.65,28.0,154,184002.0


--- Inbound units by month and warehouse (last 12 months) ---


WarehouseName,Asia-Pacific Hub,Euro Hub,North America Hub
Month,,,
2025-12,1968.0,12809.0,7727.0
2026-01,9960.0,6584.0,8832.0
2026-02,5889.0,10848.0,6992.0
2026-03,8123.0,8972.0,10231.0
2026-04,5665.0,3467.0,5041.0
2026-05,8814.0,4418.0,1950.0
2026-06,5223.0,2732.0,9484.0
2026-07,9708.0,12755.0,10322.0
2026-08,3709.0,3441.0,3607.0


In [17]:
#Task 8: Analyze shipping delays

sh = (df["shipment_tracking"]
      .merge(df["carriers"], on="CarrierID", how="left")
      .merge(wh[["WarehouseID", "WarehouseName"]]
             .rename(columns={"WarehouseID": "OriginWarehouseID",
                              "WarehouseName": "OriginWarehouse"}),
             on="OriginWarehouseID", how="left"))
sh[["CarrierName", "OriginWarehouse"]] = sh[["CarrierName", "OriginWarehouse"]].fillna("Unknown")
known = sh[sh["Status"] != "Unknown"]

print("--- Shipment status (%) ---")
print((known["Status"].value_counts(normalize=True) * 100).round(1).to_string())

print("\n--- Status mix by carrier (%) ---")
display((pd.crosstab(known["CarrierName"], known["Status"], normalize="index") * 100).round(1))

print("--- Status mix by origin warehouse (%) ---")
display((pd.crosstab(known["OriginWarehouse"], known["Status"], normalize="index") * 100).round(1))

print("--- Shipments and % delayed by month (last 12) ---")
m = known.dropna(subset=["ShipDate"]).copy()
m["Month"] = m["ShipDate"].dt.to_period("M")
display(m.groupby("Month").agg(shipments=("ShipmentID", "size"),
                               delayed_pct=("Status", lambda s: round((s == "Delayed").mean() * 100, 1))).tail(12))

--- Shipment status (%) ---
Status
In Transit    26.2
Delivered     25.4
Delayed       25.1
Processing    23.3

--- Status mix by carrier (%) ---


Status,Delayed,Delivered,In Transit,Processing
CarrierName,,,,
ChocoExpress,25.8,25.0,25.5,23.7
Global Confections Freight,24.9,25.1,26.6,23.5
RapidRoute,25.0,24.1,26.5,24.4
SweetHaul Logistics,25.3,27.3,25.9,21.5
Unknown,22.0,25.7,27.7,24.6


--- Status mix by origin warehouse (%) ---


Status,Delayed,Delivered,In Transit,Processing
OriginWarehouse,,,,
Asia-Pacific Hub,25.8,26.1,25.0,23.1
Euro Hub,23.6,25.5,26.9,24.0
North America Hub,25.4,24.4,26.6,23.6
Unknown,29.2,27.0,27.0,16.9


--- Shipments and % delayed by month (last 12) ---


,shipments,delayed_pct
Month,,
2025-12,125,26.4
2026-01,146,22.6
2026-02,118,26.3
2026-03,125,23.2
2026-04,136,23.5
2026-05,110,33.6
2026-06,118,25.4
2026-07,141,27.0
2026-08,121,24.8


In [18]:
#Task 9: Analyze demand trends

sales = df["sales_orders"].copy()
sales["Month"] = sales["OrderDate"].dt.to_period("M")

monthly = sales.groupby("Month").agg(orders=("SalesOrderID", "count"),
                                     units=("Quantity", "sum"))
monthly["change_vs_prev_month_%"] = (monthly["units"].pct_change() * 100).round(1)
print("--- Monthly demand ---")
print(monthly.to_string())

print("\n--- Demand by product ---")
display(sales.groupby("ProductName").agg(orders=("SalesOrderID", "count"),
                                         units=("Quantity", "sum"))
        .sort_values("units", ascending=False))

print("--- Units by year ---")
display(sales.groupby(sales["OrderDate"].dt.year)["Quantity"].sum())

print("--- Procurement: units ordered by month (last 12) ---")
buy = po_lines.merge(po[["PO_ID", "OrderDate"]].drop_duplicates("PO_ID"), on="PO_ID")
display(buy.groupby(buy["OrderDate"].dt.to_period("M"))
        .agg(units=("Quantity", "sum"), spend=("LineTotal", "sum")).tail(12).round(0))


--- Monthly demand ---
         orders    units  change_vs_prev_month_%
Month                                           
2024-01     116  11890.0                     NaN
2024-02     121  12262.0                     3.1
2024-03     141  15094.0                    23.1
2024-04     110  11368.0                   -24.7
2024-05     128  13348.0                    17.4
2024-06     127  13493.0                     1.1
2024-07     110  12660.0                    -6.2
2024-08     126  12696.0                     0.3
2024-09     128  12768.0                     0.6
2024-10     130  14190.0                    11.1
2024-11     108  12437.0                   -12.4
2024-12     109  11921.0                    -4.1
2025-01     153  16846.0                    41.3
2025-02     119  12105.0                   -28.1
2025-03     136  13229.0                     9.3
2025-04     123  13089.0                    -1.1
2025-05     139  14317.0                     9.4
2025-06     118  12163.0                   -15

,orders,units
ProductName,,
Enchanted Milk Chocolate,1142,121534.0
Golden Cocoa Truffles,1096,116725.0
Mystic White Chocolate,1083,113344.0
Magical Dark Chocolate,1038,105856.0


--- Units by year ---


OrderDate
2024    154127.0
2025    162248.0
2026    141084.0
Name: Quantity, dtype: float64

--- Procurement: units ordered by month (last 12) ---


,units,spend
OrderDate,,
2025-12,72123.0,659706.0
2026-01,78655.0,733585.0
2026-02,76394.0,768044.0
2026-03,58614.0,582507.0
2026-04,69328.0,567604.0
2026-05,78318.0,757744.0
2026-06,99458.0,770822.0
2026-07,68348.0,545549.0
2026-08,92091.0,827133.0


In [19]:
#Task 10: Business insights (numbers first)

kpis = {
    "Total units in stock": int(inv["QuantityOnHand"].sum()),
    "Stock value": round(inv["StockValue"].sum(), 2),
    "Total procurement spend (all PO lines)": round(pol["LineTotal"].sum(), 2),
    "Suppliers": len(df["suppliers"]),
    "Purchase orders": po["PO_ID"].nunique(),
    "Avg planned supplier lead time (days)": round(po["PlannedLeadDays"].mean(), 1),
    "Sales orders": sales["SalesOrderID"].nunique(),
    "Total units demanded": int(sales["Quantity"].sum()),
    "Peak demand month": str(monthly["units"].idxmax()),
}
display(pd.Series(kpis, name="value").to_frame())

,value
Total units in stock,54643
Stock value,377284.2
Total procurement spend (all PO lines),29192117.55
Suppliers,4
Purchase orders,1062
Avg planned supplier lead time (days),29.4
Sales orders,4339
Total units demanded,457459
Peak demand month,2025-01
